#### Modifying Linear Search (WASM)

The following cells are from the course notes:

In [ ]:
def runwasm(wasmfile):
    from IPython.display import display, Javascript
    with open(wasmfile, "rb") as file: 
        wasm_bytes = file.read() # read the WASM file and convert its content to a comma-separated string of byte values
        wasm_byte_array_str = ','.join(str(byte) for byte in wasm_bytes)
 
    display(Javascript(f""" // Javascript code to compile and run the WASM module
    const wasmBinary = new Uint8Array([{wasm_byte_array_str}]); // convert back to binary representation
 
    const params = {{
        'P0lib': {{
            'write': i => element.append(i + ' '),
            'writeln': () => element.append(document.createElement('br')),
            'read': () => window.prompt()
        }}
    }};
 
    WebAssembly.compile(wasmBinary.buffer) // compile shareable code
        .then(module => WebAssembly.instantiate(module, params)) // create an instance with memory
        // .then(instance => instance.exports.program()); // run the main program; not needed if a start function is specified
    """))

In [ ]:
def runpywasm(wasmfile):
    from pywasm.core import Machine, Runtime, FuncType, ValType
    # import pywasm; pywasm.log.lvl = 1 # for debugging
    # P0lib implementation in Python
    def write(_: Machine, args: list[int]) -> list[int]:
        print(args[0]); return []
    def writeln(_: Machine, args: list[int]) -> list[int]:
        print(); return []
    def read(_: Machine, args: list[int]) -> list[int]:
        return [int(input())]
    # Create runtime
    runtime = Runtime()
    runtime.imports = {'P0lib':
        {'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
         'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
         'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read)}}
    # Create and run instance
    instance = runtime.instance_from_file(wasmfile)

In [ ]:
def runwasmtime(wasmfile):
    from wasmtime import Engine, Store, Module, Linker, Func, FuncType, ValType
    # P0lib implementation in Python
    def write(i: int): print(i)
    def writeln(): print()
    def read() -> int: return int(input())
    # Create engine, store, and module
    engine = Engine()
    store = Store(engine)
    module = Module(store.engine, open(wasmfile, 'rb').read())
    # Use Linker to create the P0lib library
    write_func = Func(store, FuncType([ValType.i32()], []), write)
    writeln_func = Func(store, FuncType([], []), writeln)
    read_func = Func(store, FuncType([], [ValType.i32()]), read)
    linker = Linker(engine)
    linker.define(store, "P0lib", "write", write_func)
    linker.define(store, "P0lib", "writeln", writeln_func)
    linker.define(store, "P0lib", "read", read_func)
    # Create and run instance
    instance = linker.instantiate(store, module)

---

Compile the P0 program below by running the next two cells:

In [ ]:
import import_ipynb
from P0 import compileString

In [ ]:
compileString("""
const N = 5
var a: [1 .. N] → integer
program linearsearch
    var i, x: integer
        x ← read(); i := 0 // read number to be searched
        while i < N do i := i + 1; a[i] ← read()  // read array elements
        while (i > 0) and (a[i] ≠ x) do i := i - 1
        writeln(); write(i) // write index of last occurrence of x or 0 if not found
""", 'linearsearch.wat')

You can convert `linearsearch.wat` to a binary file and run it using one of the three methods above for running WebAssembly in Jupyter:

In [ ]:
!wat2wasm linearsearch.wat

In [ ]:
runwasm("linearsearch.wasm")

#### runpywasm("linearsearch.wasm")

In [ ]:
runwasmtime("linearsearch.wasm")

Now inspect the generated code by running the cell below or leaving out the last parameter of `compileString` and using `print` to print the resulting string. Explain the role of each line!

In [ ]:
!cat linearsearch.wat

YOUR ANSWER HERE

Perform the following modifications of the above program; it is repeated below:

1. Make variables `i` and `x` global rather than local to the main program. What changes do you observe in the generated code?
2. Make array `a` a local variable of the main program. What changes do you observe in the generated code?
3. Make constant `N` a local constant of the main program. What changes do you observe in the generated code?

In [ ]:
wasm = compileString("""
const N = 5
var a: [1 .. N] → integer
program linearsearch
    var i, x: integer
        x ← read(); i := 0 // read number to be searched
        while i < N do i := i + 1; a[i] ← read()  // read array elements
        while (i > 0) and (a[i] ≠ x) do i := i - 1
        writeln(); write(i) // write index of last occurrence of x or 0 if not found
""")
print(wasm)

YOUR ANSWER HERE